In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# 1. モデル方程式の定義
def model(z, t, params):
    x1, y1, x2, y2 = z
    r_x1, r_y1, r_x2, r_y2 = params['r']
    lam = params['lambda'] # 行列 [[l11, l12], [l21, l22]]
    c = params['c']
    d = params['d']
    
    # 係数の展開 (ユーザーの定義に基づく)
    l11, l12 = lam[0]
    l21, l22 = lam[1]
    c1, c2 = c
    d1, d2 = d
    
    # 修正された方程式
    dx1dt = -r_x1*x1 + c1*l11*x1*y1 + d1*l21*x1*y2
    dy1dt =  r_y1*y1 -    l11*x1*y1 -    l12*x2*y1
    dx2dt = -r_x2*x2 + c2*l12*x2*y1 + d2*l22*x2*y2
    dy2dt =  r_y2*y2 -    l21*x1*y2 -    l22*x2*y2
    
    return [dx1dt, dy1dt, dx2dt, dy2dt]

# 2. パラメータ設定 (共存しそうな値を設定)
# ※密度効果なしモデルは初期値やパラメータに敏感で発散しやすいため
#   比較的小さな相互作用係数を設定しています。
params = {
    'r': [1.0, 2.0, 1.0, 2.0],      # [rx1, ry1, rx2, ry2]
    'lambda': [[0.8, 0.2],          # [[l11, l12],
               [0.2, 0.8]],         #  [l21, l22]]
    'c': [0.8, 0.8],                # [c1, c2]
    'd': [0.8, 0.8]                 # [d1, d2]
}

# 初期状態 (平衡点から少しずらした位置)
z0 = [2.0, 2.0, 1.5, 1.5] # [x1, y1, x2, y2]

# 時間軸
t = np.linspace(0, 50, 5000)

# 3. 微分方程式を解く
z = odeint(model, z0, t, args=(params,))

x1 = z[:, 0]
y1 = z[:, 1]
x2 = z[:, 2]
y2 = z[:, 3]

# パラメータの展開
r_list = params['r']
lambda_matrix = params['lambda']
c_list = params['c']
d_list = params['d']

# タイトル文字列の生成 (LaTeX表記を使用)
full_title = (
    f'4 Species Lotka-Volterra Dynamics\n'
    # rパラメータ
    f'$r_{{x1}}$={r_list[0]}, $r_{{y1}}$={r_list[1]}, $r_{{x2}}$={r_list[2]}, $r_{{y2}}$={r_list[3]}\n'
    # c, d パラメータ
    f'$c_1$={c_list[0]}, $d_1$={d_list[0]} | $c_2$={c_list[1]}, $d_2$={d_list[1]}\n'
    # lambda パラメータ
    f'$\\lambda_{{11}}$={lambda_matrix[0][0]}, $\\lambda_{{12}}$={lambda_matrix[0][1]}, $\\lambda_{{21}}$={lambda_matrix[1][0]}, $\\lambda_{{22}}$={lambda_matrix[1][1]}'
)

# 4. 可視化
plt.figure(figsize=(14, 6))

# --- 全体のタイトルを追加 (最も大きなタイトル) ---
plt.suptitle(
    full_title,
    fontsize=14, # フォントサイズを大きく設定
    fontweight='bold',
    y=1.03 # 縦位置を少し上に調整
)

# --- 図1: 時系列グラフ ---
plt.subplot(1, 2, 1)
plt.plot(t, x1, label='$x_1$ (Predator 1)')
plt.plot(t, y1, label='$y_1$ (Prey 1)', alpha=0.7)
plt.plot(t, x2, label='$x_2$ (Predator 2)', linestyle='--')
plt.plot(t, y2, label='$y_2$ (Prey 2)', alpha=0.7, linestyle='--')
plt.xlabel('Time')
plt.ylabel('Population')
plt.title('Time Series')
plt.legend()
plt.grid(True)

# --- 図2: 相図 (ペアごとの射影) ---
plt.subplot(1, 2, 2)
plt.plot(y1, x1, label='Species 1 ($y_1$ vs $x_1$)')
plt.plot(y2, x2, label='Species 2 ($y_2$ vs $x_2$)')
plt.xlabel('Prey Population ($y$)')
plt.ylabel('Predator Population ($x$)')
plt.title('Phase Portrait (2D Projection)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. パラメータ設定
params = {
    'r': [1.0, 1.0, 1.0, 1.0],      # [rx1, ry1, rx2, ry2]
    'lambda': [[0.5, 0.2],          # [[l11, l12],
               [0.2, 0.5]],         #  [l21, l22]]
    'c': [0.8, 0.8],                # [c1, c2]
    'd': [0.8, 0.8]                 # [d1, d2]
}

# パラメータ展開
r_x1, r_y1, r_x2, r_y2 = params['r']
l11, l12 = params['lambda'][0]
l21, l22 = params['lambda'][1]
c1, c2 = params['c']
d1, d2 = params['d']

# 2. 平衡点 (x*, y*) の計算 (前回の数式を使用)
# 行列式
det_x = l11 * l22 - l12 * l21
det_y = c1 * d2 * l11 * l22 - c2 * d1 * l12 * l21

# 捕食者 x*
x1_star = (l22 * r_y1 - l12 * r_y2) / det_x
x2_star = (l11 * r_y2 - l21 * r_y1) / det_x

# 被食者 y*
y1_star = (d2 * l22 * r_x1 - d1 * l21 * r_x2) / det_y
y2_star = (c1 * l11 * r_x2 - c2 * l12 * r_x1) / det_y

print(f"Equilibrium Point: x1={x1_star:.2f}, y1={y1_star:.2f}, x2={x2_star:.2f}, y2={y2_star:.2f}")

# 3. グリッドの作成 (平衡点を中心に表示範囲を決める)
range_span = 1.0 # 表示幅
x1_vals = np.linspace(max(0, x1_star - range_span), x1_star + range_span, 20)
y1_vals = np.linspace(max(0, y1_star - range_span), y1_star + range_span, 20)
X1, Y1 = np.meshgrid(x1_vals, y1_vals)

# 4. ベクトル場 (dx1/dt, dy1/dt) の計算
# 注意: ここでは x2, y2 を平衡値 (x2_star, y2_star) に固定して断面を見る
U = -r_x1*X1 + c1*l11*X1*Y1 + d1*l21*X1*y2_star
V =  r_y1*Y1 -    l11*X1*Y1 -    l12*x2_star*Y1

# 正規化 (矢印の長さを揃えて見やすくする)
Norm = np.sqrt(U**2 + V**2)
U_norm = U / Norm
V_norm = V / Norm

# 5. ヌルクラインの計算 (x2=x2*, y2=y2* とした時の線)
# dx1/dt = 0 => x1(-r_x1 + c1*l11*y1 + d1*l21*y2*) = 0
# y1 について解くと: y1 = (r_x1 - d1*l21*y2*) / (c1*l11)
nullcline_x1 = (r_x1 - d1 * l21 * y2_star) / (c1 * l11)

# dy1/dt = 0 => y1(r_y1 - l11*x1 - l12*x2*) = 0
# x1 について解くと: x1 = (r_y1 - l12*x2*) / l11
nullcline_y1 = (r_y1 - l12 * x2_star) / l11

# --- プロット作成 ---
plt.figure(figsize=(10, 8))

# ストリームプロット (流れの線)
plt.streamplot(X1, Y1, U, V, color='lightgray', density=1.5)

# クイーバープロット (方向矢印)
plt.quiver(X1, Y1, U_norm, V_norm, angles='xy', scale_units='xy', scale=10, color='blue', alpha=0.6, width=0.003)

# ヌルクラインの描画
# x1-nullcline (dx1/dt=0): 水平線 (y1 = constant)
plt.axhline(y=nullcline_x1, color='green', linestyle='--', linewidth=2, label='$x_1$-nullcline ($\dot{x}_1=0$)')

# y1-nullcline (dy1/dt=0): 垂直線 (x1 = constant)
plt.axvline(x=nullcline_y1, color='orange', linestyle='--', linewidth=2, label='$y_1$-nullcline ($\dot{y}_1=0$)')

# 平衡点のプロット
plt.plot(x1_star, y1_star, 'ro', markersize=10, label='Equilibrium Point', zorder=5)

plt.xlabel('$x_1$ (Predator 1)')
plt.ylabel('$y_1$ (Prey 1)')
plt.title('Phase Plane Analysis (Slice at $x_2^*, y_2^*$)\nNeutrally Stable Center')
plt.legend()
plt.grid(True)
plt.xlim(x1_vals[0], x1_vals[-1])
plt.ylim(y1_vals[0], y1_vals[-1])

plt.show()

In [ ]:
output_filename = '4種の時系列グラフと相平面(x1 vs y1,x2 vs y2).png'
plt.savefig(output_filename)
plt.clf()

print(f"処理が完了しました。グラフを '{output_filename}' として保存しました。")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# 1. モデル方程式の定義
def model(z, t, params):
    x1, y1, x2, y2 = z
    r_x1, r_y1, r_x2, r_y2 = params['r']
    lam = params['lambda'] # 行列 [[l11, l12], [l21, l22]]
    c = params['c']
    d = params['d']
    
    # 係数の展開 (ユーザーの定義に基づく)
    l11, l12 = lam[0]
    l21, l22 = lam[1]
    c1, c2 = c
    d1, d2 = d
    
    # 修正された方程式
    dx1dt = -r_x1*x1 + c1*l11*x1*y1 + d1*l21*x1*y2
    dy1dt =  r_y1*y1 -    l11*x1*y1 -    l12*x2*y1
    dx2dt = -r_x2*x2 + c2*l12*x2*y1 + d2*l22*x2*y2
    dy2dt =  r_y2*y2 -    l21*x1*y2 -    l22*x2*y2
    
    return [dx1dt, dy1dt, dx2dt, dy2dt]

# 2. パラメータ設定 (共存しそうな値を設定)
# ※密度効果なしモデルは初期値やパラメータに敏感で発散しやすいため
#   比較的小さな相互作用係数を設定しています。
params = {
    'r': [1.0, 1.0, 1.0, 1.0],      # [rx1, ry1, rx2, ry2]
    'lambda': [[0.1, 0.3],           # [[l11, l12],
               [0.3, 0.1]],          #  [l21, l22]]
    'c': [0.8, 0.6],                  # [c1, c2]
    'd': [0.6, 0.8]                   # [d1, d2]
}

# 初期状態 (平衡点から少しずらした位置)
z0 = [2.0, 2.0, 1.5, 1.5] # [x1, y1, x2, y2]

# 時間軸
t = np.linspace(0, 1000, 5000)

# 3. 微分方程式を解く
z = odeint(model, z0, t, args=(params,))

x1 = z[:, 0]
y1 = z[:, 1]
x2 = z[:, 2]
y2 = z[:, 3]

# パラメータの展開
r_list = params['r']
lambda_matrix = params['lambda']
c_list = params['c']
d_list = params['d']

# タイトル文字列の生成 (LaTeX表記を使用)
full_title = (
    f'4 Species Lotka-Volterra Dynamics\n'
    # rパラメータ
    f'$r_{{x1}}$={r_list[0]}, $r_{{y1}}$={r_list[1]}, $r_{{x2}}$={r_list[2]}, $r_{{y2}}$={r_list[3]}\n'
    # c, d パラメータ
    f'$c_1$={c_list[0]}, $d_1$={d_list[0]} | $c_2$={c_list[1]}, $d_2$={d_list[1]}\n'
    # lambda パラメータ
    f'$\\lambda_{{11}}$={lambda_matrix[0][0]}, $\\lambda_{{12}}$={lambda_matrix[0][1]}, $\\lambda_{{21}}$={lambda_matrix[1][0]}, $\\lambda_{{22}}$={lambda_matrix[1][1]}'
)

# 4. 可視化
plt.figure(figsize=(14, 6))

# --- 全体のタイトルを追加 (最も大きなタイトル) ---
plt.suptitle(
    full_title,
    fontsize=14, # フォントサイズを大きく設定
    fontweight='bold',
    y=1.03 # 縦位置を少し上に調整
)


# --- 図2: 軌道図 (Phase Portraits) (追加) ---
plt.subplot(1, 2, 2)
# 捕食者1 ($x_1$) 対 獲物1 ($y_1$)
plt.plot(y1, x2, label='Prey 1 ($y_1$) vs Predator 1 ($x_2$)', color='blue')

plt.xlabel('Prey Population ($y$)')
plt.ylabel('Predator Population ($x$)')
plt.title('Phase Portrait (Orbit Diagram)')
plt.legend()
plt.grid(True)

# --- レイアウト調整と保存 (plt.show()をplt.savefig()に置き換え) ---
# suptitleが表示領域外にならないように調整
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('lotka_volterra_dynamics_phase_and_time_series.png')